## **Fine Tuning LLM**

Nesta aula vamos abordar o fine tuning de modelos LLM usando métodos que simplificam o fine tuning, técnicas para reduzir o tamanho dos modelos (quantizados) e o treinamento usando PEFT e LoRA para ajustar uma pequena parte do modelo.

## Para os não iniciados ainda no mundo de LLM

Aqui vai o glossário dos termos que vamos usar aqui para deixar todos na mesma página.



*   **Large Language Model (LLM)** - Preditor de palavras, simples assim. A diferença de uma LLM para o auto completar do celular é a quantidade de dados, que na LLM a quantidade é gigante e o modelo é treinado reconhecendo sequências de palavras comuns e preenchendo lacunas.

*   **Attention** - Imagine ler uma frase e entender quais são as palavras anteriores mais relevantes/importantes para entender cada nova palavra, isso é atenção. Ao invés de ler a frase inteira, attention permite que cada palavra pondere o quanto deve considerar todas as outras.

*    **Parameter** - Um número dentro de um modelo que pode mudar durante o aprendizado, ajustado conforme o treinamento.

*    **Weights** - Geralmente sinônimo de parâmetros que controlam o quão fortemente uma parte da entrada afeta a saída

*    **Bias** - Um pequeno número extra adicionado para que o modelo possa deslocar as saídas para cima ou para baixo, como um ajuste de baseline.

*    **Transformer** - Um Transformer é um modelo especial construído em torno da atenção, permitindo que ela "veja" cada palavra de uma frase em paralelo, em vez de uma por uma. É como um grupo de estudos onde todos leem a redação inteira de uma só vez e depois discutem quais frases são mais importantes para a ideia principal. O transfomer é a espinha dorsal das LLMs.

*    **Quantized** - Redução da precisão dos pesos (por exemplo, 16 bits → 4 bits) para reduzir o uso de memória, com perda mínima de precisão.

*    **PEFT** - ( Ajuste Fino Eficiente em Parâmetros ) — atualizando apenas pequenas camadas adaptadoras em vez de todo o modelo.

*    **LoRA** - Um atalho inteligente para ensinar novos truques a um modelo de IA enorme, ajustando apenas uma pequena parte dele, em vez de retreinar tudo. Veja assim: você "congela" a grande maioria dos parâmetros existentes do modelo e insere duas matrizes pequenas e treináveis ​​(pense nelas como complementos leves) em cada camada. Durante o ajuste fino, apenas esses complementos aprendem, reduzindo drasticamente o tempo e o custo computacional.

*    **LoRA "r"** - Classificação (tamanho) do adaptador. Um r mais alto oferece mais capacidade, mas usa mais memória.

*    **LoRA α (alfa)** - Um fator de escala para atualizações do adaptador — como um "botão de volume" para aprender a intensidade.

*    **Token** - Um token é um pequeno pedaço de texto (regra geral ~ aproximadamente 4–5 caracteres) — uma palavra, parte de uma palavra, sinal de pontuação ou símbolo — que serve como unidade básica que um modelo processa.



In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets gradio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [ ]:
import torch
import os
from datasets import load_dataset
import gradio as gr
from google.colab import drive
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer

Configurações para a GPU T4 do Colab.

Otimiza o uso de memória da GPU, crucial para modelos grandes

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Nenhuma GPU detectada. O Fine-Tuning pode ser muito lento.")

Nenhuma GPU detectada. O Fine-Tuning pode ser muito lento.


MONTAR GOOGLE DRIVE E VARIÁVEIS (Para Salvar o Modelo)

Execute esta célula para montar seu Google Drive.

Diretório onde o modelo LoRA será salvo (e carregado)

In [ ]:
LORA_PATH = "/content/drive/MyDrive/llama_finetune/adaptadores_lora"

print("Montando Google Drive...")
drive.mount('/content/drive')
os.makedirs(LORA_PATH, exist_ok=True)
print(f"Diretório de destino do modelo: {LORA_PATH}")

Montando Google Drive...
Mounted at /content/drive
Diretório de destino do modelo: /content/drive/MyDrive/llama_finetune/adaptadores_lora


**DEFINIÇÃO DE VARIÁVEIS E HYPERPARÂMETROS**

Usaremos um modelo pequeno da família Llama (7B) otimizado para chat/instrução.
Usaremos o dataset da Hugging Face chamado timdettmers/openassistant-guanaco para o Fine-Tuning. Este dataset é ótimo para tarefas de instrução/QA.

**Configurações do PEFT (LoRA)**


*   r: rank (matriz de rank baixo), dimension of the update matrices. Valores típicos: 8, 16, 32, 64.

*   lora_alpha: Scaling factor for the learned weights. Valores típicos: 16, 32, 64.

*   lora_dropout: Taxa de dropout (regularização)


**# Hyperparâmetros de Treinamento**

*   num_train_epochs: Número de vezes que o modelo verá o dataset (1 é comum para Fine-Tuning)
`num_train_epochs = 1`

*    per_device_train_batch_size: Tamanho do lote de treinamento (depende da VRAM) `per_device_train_batch_size = 4`

*    gradient_accumulation_steps: Acumula gradientes para simular um lote maior
`gradient_accumulation_steps = 4`

*    gradient_checkpointing: Economiza VRAM à custa de velocidade de treinamento `gradient_checkpointing = True`

*    max_grad_norm: Limita o valor máximo do gradiente para evitar gradientes explosivos `max_grad_norm = 0.3`

*    learning_rate: Taxa de aprendizado `learning_rate = 2e-4`

*    weight_decay: Regularização `weight_decay = 0.001`

*    optim: Otimizador (paged_adamw_32bit é otimizado para memória) `optim = "paged_adamw_32bit"`

*    lr_scheduler_type: Tipo de agendador de taxa de aprendizado `lr_scheduler_type = "cosine"`

*    max_seq_length: Comprimento máximo da sequência de tokens (depende do modelo) `max_seq_length = 5384`








In [ ]:
model_name = "NousResearch/Llama-2-7b-chat-hf"
dataset_name = "timdettmers/openassistant-guanaco"
MAX_SAMPLES = 1000
# Nome do novo modelo que será salvo (após o Fine-Tuning)
new_model = "llama-2-7b-timdettmers-finetune"
# Diretório para salvar os resultados do treinamento
output_dir = "./results"

lora_r = 32
lora_alpha = 16
lora_dropout = 0.1

num_train_epochs = 1
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_seq_length = 5384

QUANTIZAÇÃO (4-bit) E CARREGAMENTO DO MODELO

Configuração de Quantização com BitsAndBytes (QLoRA)
Esta configuração permite carregar o modelo em 4 bits, reduzindo drasticamente o uso de VRAM

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Normal Float 4-bit (Melhor para LLMs)
    bnb_4bit_compute_dtype=torch.float16, # Tipo de dado para computação
    bnb_4bit_use_double_quant=False, # Não usamos dupla quantização aqui
)

In [ ]:
# Carregar Modelo Base
print("Carregando Modelo Base e Tokenizer (com Quantização 4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0} # Mapeia para a GPU 0
)
model.config.use_cache = False
model.config.pretraining_tp = 1 # Required for Llama 2

# Carregar Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Necessário para treinamento

# Preparar o Modelo para Treinamento k-bit (PEFT)
# Habilita gradientes no modelo quantizado, essencial para o LoRA funcionar
model = prepare_model_for_kbit_training(model)

print("Modelo base carregado e quantizado com sucesso!")

PREPARAÇÃO E AMOSTRAGEM DO DATASET (Pequeno para rodar Rápido)

In [ ]:
# Carregar dataset
print(f"Carregando Dataset: {dataset_name}")
try:
    # Carrega o dataset e seleciona o split de treino
    dataset = load_dataset(dataset_name, split="train")

    if len(dataset) > MAX_SAMPLES:
        dataset = dataset.select(range(MAX_SAMPLES))

except Exception as e:
    print(f"ERRO ao carregar o dataset. Mensagem: {e}")
    raise

print("Dataset carregado e amostrado com sucesso.")
print(f"Número de amostras para treinamento: {len(dataset)}")
print("-" * 20)
print("Exemplo de entrada (Formato de Diálogo de Instrução):")
print(dataset[0]['text'])
print("-" * 20)

CONFIGURAÇÃO DO LO-RA (PEFT)

In [ ]:
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj",
    ],
)

CONFIGURAÇÃO E EXECUÇÃO DO TREINAMENTO (SFTTrainer)

In [ ]:
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=200,
    logging_steps=20,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=True, # Usa precisão mista (float16)
    bf16=False,
    max_grad_norm=max_grad_norm,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Inicializa o SFTTrainer (Supervised Fine-Tuning Trainer)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    #tokenizer=tokenizer,
    args=training_arguments,
)

# Inicia o treinamento
print("\n" + "="*50)
print(f"INICIANDO FINE-TUNING (LoRA/QLoRA) em {len(dataset)} amostras.")
print("Este processo deve durar apenas alguns minutos com a GPU T4.")
print("="*50)
trainer.train()

Salva o modelo final (apenas os adaptadores LoRA)

In [ ]:
trainer.model.save_pretrained(LORA_PATH)
print(f"\nFine-Tuning concluído! Adaptadores LoRA salvos em: {LORA_PATH}")

AVALIAÇÃO E TESTE DE INFERENCIA

In [ ]:
logging.set_verbosity(logging.CRITICAL)

# Exemplo de prompt que o modelo deve responder após o Fine-Tuning
prompt_test = "Como faço para calcular a média e o desvio padrão de um conjunto de dados no Python?"
prompt_input = f"""### Human:
{prompt_test}
### Assistant:
"""
print("\n" + "="*50)
print("TESTE ANTES DO FINE-TUNING (Modelo Llama 2 Base)")
print("="*50)

try:
    # O modelo base pode não dar uma resposta de alta qualidade para o formato de instrução.
    pipe_base = pipeline(task="text-generation", model=model_name, tokenizer=tokenizer, max_length=200)
    result_base = pipe_base(prompt_input)
    print(f"Resposta Gerada (BASE):\n{result_base[0]['generated_text']}")
except Exception as e:
    print(f"Não foi possível executar o pipeline BASE. Erro: {e}")

In [ ]:
print("\n" + "="*50)
print("CONFIGURANDO O AGENTE (CARREGAMENTO DO MODELO FINAL)")
print("="*50)

# --- NOVO: VERIFICAÇÃO DE ARQUIVOS SALVOS ---
adapter_config_path = os.path.join(LORA_PATH, "adapter_config.json")
if not os.path.exists(adapter_config_path):
    print(" ERRO: Arquivo 'adapter_config.json' não encontrado no Google Drive.")
    print("Isso geralmente significa que a etapa de Fine-Tuning (Seção 4) falhou ou não foi concluída.")
    print(f"Caminho verificado: {LORA_PATH}")
    # Geramos um erro claro para interromper a execução.
    raise FileNotFoundError(f"Arquivos do modelo PEFT/LoRA não encontrados em {LORA_PATH}. Por favor, execute a Seção 4 novamente.")
# --- FIM VERIFICAÇÃO ---

# 1. Configuração QLoRA (Repetida para o carregamento)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
)

# 2. Recarregar Tokenizer e Modelo Base
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model_base_full = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map={"": 0}
)

# 3. Carregar Adaptadores LoRA (Do Google Drive)
try:
    model_peft = PeftModel.from_pretrained(model_base_full, LORA_PATH)
    print("Adaptadores LoRA carregados com sucesso!")
except Exception as e:
    print(f"ERRO ao carregar adaptadores LoRA do Drive: {e}")
    print("Verifique se o Fine-Tuning foi concluído e se o caminho do Drive está correto.")
    raise

# Cria o pipeline de geração (Modelo Pós-Fine-Tuning)
pipe_tuned = pipeline(
    task="text-generation",
    model=model_peft,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    # REMOVIDO: o argumento 'device=device' foi removido para evitar o conflito
    # com o gerenciamento de dispositivos do 'accelerate' (usado no QLoRA).
)
print("Pipeline de inferência do Agente criado com sucesso!")

In [ ]:
# Limpeza de VRAM
del model, trainer, model_base_full, model_peft
torch.cuda.empty_cache()
print("\nLimpeza de VRAM concluída.")

INTERFACE DE CHAT COM GRADIO


In [ ]:
def generate_response(message, history):
    """Gera a resposta do Agente baseado no histórico de chat e na nova mensagem."""

    # Formato de Diálogo (O mesmo formato usado no Fine-Tuning)
    formatted_prompt = f"### Human:\n{message}\n\n### Assistant:\n"

    try:
        # Geração da resposta usando o pipeline afinado
        result = pipe_tuned(
            formatted_prompt,
            max_new_tokens=200,
            do_sample=True,
            top_k=10,
            num_return_sequences=1,
            eos_token_id=tokenizer.eos_token_id,
        )

        # Extrai e formata a resposta
        full_text = result[0]['generated_text']

        # Remove a parte do prompt para retornar apenas a resposta do Assistente
        response_text = full_text.split("### Assistant:\n")[-1].strip()

        return response_text

    except Exception as e:
        return f"Desculpe, houve um erro na geração da resposta: {e}"

# Cria a Interface do Gradio
print("\n" + "="*50)
print("INICIANDO CHATBOT COM GRADIO")
print("O link público será gerado abaixo.")
print("="*50)

# Configurações visuais e de interação do chat
chat_interface = gr.ChatInterface(
    generate_response,
    title="Agente Conversacional Llama 2 (Fine-Tuned)",
    description="Faça perguntas sobre Data Science, Python ou Estatística para o modelo que acabamos de treinar.",
    textbox=gr.Textbox(placeholder="Digite sua pergunta aqui...", container=False, scale=7),
    theme="soft",
)

# Inicia o servidor Gradio
# O 'share=True' gera um link público temporário para acessar a interface
chat_interface.launch(share=True)